# 07 — Combined

Stacks the interventions that helped **on val**, then reports on test once. Nothing here looks at test before the configuration is fixed.

**Selection rule (automatic, val only).** Margin = 2 × |ball mAP50-95(02) − ball mAP50-95(02b)| on val — twice the seed noise. An intervention is stacked only if its val ball gain over its reference exceeds the margin:

| Intervention | Run | Reference |
|---|---|---|
| training resolution | 03 (960 / 1280) | best of 640 / 960 / 1280 on val |
| narrow scale jitter | 04a | 02 |
| motion blur | 04b | 04a (it was added on top of 04a) |
| P2 head | 05 | 02 |
| class-weighted loss | 05b | 02 |

SAHI (inference only) is tested on val on top of the combined weights with the same evaluator; if it clears the margin it is reported as a second row.

**Additivity (test, reporting only):** sum of the individual test gains of the stacked interventions vs the combined test gain — the plan asks whether gains add up or overlap.

In [ ]:
SMOKE = False  # True: 1 epoch on 25% of train, written to <run_name>_smoke
SLICE_SIZES = [640, 960]

In [ ]:
import sys, json
sys.path.insert(0, "/content/ballhawk")
import pandas as pd
import albumentations as A
import ballhawk_common as bc

def val_ball(run):
    v = json.loads((bc.RESULTS_DIR / run / "val_metrics.json").read_text())
    return next(c for c in v["per_class"] if c["class"] == "ball")["mAP50_95"]

def test_ball(run):
    r = json.loads((bc.RESULTS_DIR / run / "metrics.json").read_text())
    return next(c for c in r["per_class"] if c["class"] == "ball")["mAP50_95"]

BASE = "baseline_yolo11s_640"
margin = 2 * abs(val_ball(BASE) - val_ball("baseline_yolo11s_640_seed1"))
res = {640: BASE, 960: "res_yolo11s_960", 1280: "res_yolo11s_1280"}
imgsz = max(res, key=lambda s: val_ball(res[s]))
checks = [  # (name, run, reference run, train overrides if stacked)
    ("scale 0.2", "aug_scale02_yolo11s_640", BASE, {"scale": 0.2}),
    ("motion blur", "aug_scale02_mblur_yolo11s_640", "aug_scale02_yolo11s_640",
     {"augmentations": bc.default_albumentations() + [A.MotionBlur(blur_limit=(3, 7), p=0.3)]}),
    ("P2 head", "p2_yolo11s_640", BASE, None),
    ("cls_pw 0.5", "clsw05_yolo11s_640", BASE, {"cls_pw": 0.5}),
]
rows, overrides, stacked = [], {}, []
rows.append({"intervention": f"imgsz {imgsz}", "val_gain": val_ball(res[imgsz]) - val_ball(BASE), "stacked": imgsz != 640})
for name, run, ref, ov in checks:
    gain = val_ball(run) - val_ball(ref)
    use = gain > margin
    rows.append({"intervention": name, "val_gain": gain, "stacked": use})
    if use:
        stacked.append((name, run))
        if ov is None:
            raise NotImplementedError("P2 cleared the margin: stacking it needs the p2 config + pretrained path; decide explicitly")
        overrides.update(ov)
if "scale 0.2" not in [s[0] for s in stacked] and "augmentations" in overrides:
    overrides["scale"] = 0.2  # motion blur was only measured on top of scale 0.2
decision = pd.DataFrame(rows).round(4)
print(f"margin (2x val seed noise) = {margin:.4f}; imgsz = {imgsz}; overrides = {sorted(overrides)}")
decision

In [ ]:
batch = 8 if imgsz == 1280 else 16  # batch 16 OOMs on the T4 at 1280 (03)
if not overrides:
    # Only the resolution change cleared the margin: the combined config IS the 03 run at this imgsz.
    # Training is deterministic, so retraining would reproduce it; reuse its weights and metrics.
    run = res[imgsz]
    record = json.loads((bc.RESULTS_DIR / run / "metrics.json").read_text())
    print(f"No other intervention cleared the val margin; combined = {run} (no retraining, no new metrics.json).")
else:
    parts = ([f"imgsz {imgsz}"] if imgsz != 640 else []) + [s[0] for s in stacked]
    note = (f"Chosen on val with margin {margin:.4f} (2x seed noise). Val gains: "
            + "; ".join(f"{r.intervention} {r.val_gain:+.4f}" for r in decision.itertuples()) + ". ultralytics 8.4.163.")
    if batch != 16:
        note += f" Batch {batch} at imgsz {imgsz} (T4 memory), same as 03."
    record = bc.run_experiment(
        run_name=f"combined_yolo11s_{imgsz}", notebook="07", variable_changed=" + ".join(parts),
        model_cfg="yolo11s.pt", imgsz=imgsz, batch=batch, notes=note, smoke=SMOKE, **overrides)
    run = record["run_name"]
pd.DataFrame(record["per_class"]).set_index("class").round(3)

In [ ]:
# SAHI on top of the combined weights: decided on val with the same evaluator, then reported on test.
data_yaml = bc.prepare_data()
weights = bc.RESULTS_DIR / run / "best.pt"
def ball_of(pc): return next(c for c in pc if c["class"] == "ball")["mAP50_95"]
from ultralytics import YOLO
yolo = YOLO(weights)
full_val = ball_of(bc.evaluate_predictions({p: bc.yolo_arrays(yolo, p, imgsz) for p in bc.split_images(data_yaml, "val")}, data_yaml, "val")[1])
score_dm = bc.sahi_model(weights, 0.001, imgsz)
sweep = {s: ball_of(bc.evaluate_predictions({p: bc.sahi_arrays(score_dm, p, s) for p in bc.split_images(data_yaml, "val")}, data_yaml, "val")[1])
         for s in SLICE_SIZES}
best = max(sweep, key=sweep.get)
print(f"val ball: full-frame {full_val:.4f}; SAHI {sweep}; margin {margin:.4f}")
if sweep[best] - full_val > margin and not SMOKE:
    overall, per_class = bc.evaluate_predictions({p: bc.sahi_arrays(score_dm, p, best) for p in bc.split_images(data_yaml, "test")}, data_yaml, "test")
    time_dm = bc.sahi_model(weights, 0.25, imgsz)
    fps = bc.measure_fps(None, data_yaml, imgsz, predict=lambda p: bc.sahi_predict(time_dm, p, best))
    sahi_record = {**record, "run_name": f"sahi_{run}_s{best}", "model": f"{run} best.pt",
                   "notebook": "07", "variable_changed": f"{run} + sliced inference {best}px", "overall": overall,
                   "per_class": per_class, "inference_fps": fps, "train_minutes": 0.0,
                   "notes": f"Inference only on 07 weights; slice chosen on val {sweep} vs full-frame {full_val:.4f}. ultralytics 8.4.163."}
    print(bc.write_metrics(sahi_record, bc.RESULTS_DIR / sahi_record["run_name"]))
else:
    print("SAHI does not clear the margin on val for the combined model; not stacked.")

In [ ]:
# Additivity on test (reporting only): do the individual gains add up?
single = {"imgsz": test_ball(res[imgsz]) - test_ball(BASE) if imgsz != 640 else 0.0}
single.update({name: test_ball(r) - test_ball(BASE if name != "motion blur" else "aug_scale02_yolo11s_640") for name, r in stacked})
combined_gain = ball_of(record["per_class"]) - test_ball(BASE)
print(pd.Series(single).round(4).to_string())
print(f"sum of individual test gains {sum(single.values()):+.4f} vs combined test gain {combined_gain:+.4f}")